# Module 1 · Lesson 04: Compare Models Side by Side

A key skill for any AI engineer is **choosing the right model** for the job.
In this notebook we send the *same* prompt to multiple models and compare results.

## What you will learn
1. How to call different providers with a **unified interface**
2. Compare **quality, speed, and cost** across models
3. Build a reusable comparison framework
4. Understand the **speed vs quality vs cost** trade-off

In [1]:
# Setup
import os, time
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / ".env")

from openai import OpenAI

openai_client = OpenAI()

# Optional: Anthropic
anthropic_client = None
try:
    import anthropic
    if os.getenv("ANTHROPIC_API_KEY"):
        anthropic_client = anthropic.Anthropic()
        print("OpenAI + Anthropic clients ready")
    else:
        print("OpenAI ready | Anthropic key not set — OpenAI-only comparison")
except ImportError:
    print("OpenAI ready | anthropic not installed — OpenAI-only comparison")

OpenAI + Anthropic clients ready


---
## 1. Model Registry & Pricing

First, let's define the models and their costs (per 1M tokens, early 2025):

LiteLLM provides a **unified interface** to call any LLM provider with the same code.
Instead of learning separate SDKs, you write one `completion()` call and LiteLLM handles the rest.

```bash
pip install litellm openai anthropic
```

In [12]:
import os
import json
import litellm
from litellm import completion, completion_cost, model_cost
from IPython.display import display, Markdown

# Available models
MODELS = [
    "gpt-4o-mini",
    "gpt-5.2",
    "claude-haiku-4-5",
    "claude-opus-4-6",
]

# Check pricing from LiteLLM's built-in database
pricing_md = []

for model in MODELS:
    try:
        info = model_cost[model]
        input_price = info["input_cost_per_token"] * 1_000_000
        output_price = info["output_cost_per_token"] * 1_000_000
        pricing_md.append(
            f"- **{model}** -- Input: `${input_price:.2f}` | Output: `${output_price:.2f}`"
        )
    except KeyError:
        pricing_md.append(f"- **{model}** -- Pricing not found")

display(Markdown("### Model Pricing (per 1M tokens)\n" + "\n".join(pricing_md)))

### Model Pricing (per 1M tokens)
- **gpt-4o-mini** -- Input: `$0.15` | Output: `$0.60`
- **gpt-5.2** -- Input: `$1.75` | Output: `$14.00`
- **claude-haiku-4-5** -- Input: `$1.00` | Output: `$5.00`
- **claude-opus-4-6** -- Input: `$5.00` | Output: `$25.00`

---
### Sending the Same Prompt to Every Model

With `litellm.completion()` we can send the exact same prompt to OpenAI and Anthropic models
using **identical code**. LiteLLM translates the call to each provider's native API behind the scenes.

In [14]:
prompt = "Explain what an API is in one paragraph."

for model in MODELS:
    try:
        response = completion(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
        )

        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens
        cost = completion_cost(completion_response=response)

        md = f"""### {model}
**{reply}**

- Input tokens: `{input_tokens}` | Output tokens: `{output_tokens}` | Cost: `${cost:.6f}`
"""
        display(Markdown(md))

    except Exception as e:
        display(Markdown(f"### {model}\n`Error: {e}`"))

### gpt-4o-mini
**An **API (Application Programming Interface)** is a set of rules, protocols, and tools that allows different software applications to communicate with each other. It acts as an intermediary, defining the methods and data formats that programs can use to request and exchange information, without needing to understand each other's internal workings. For example, when you use a weather app on your phone, it sends a request through an API to a remote server, which processes the request and sends back the weather data in a structured format that the app can display. APIs are fundamental to modern software development because they enable developers to leverage existing services and functionalities—such as payment processing, mapping, or social media integration—without having to build everything from scratch, promoting efficiency, modularity, and interoperability across platforms and systems.**

- Input tokens: `16` | Output tokens: `96` | Cost: `$0.000060`


### gpt-5.2
**An **API (Application Programming Interface)** is a set of rules, protocols, and tools that allows different software applications to communicate with each other. It acts as an intermediary, defining the methods and data formats that programs can use to request and exchange information, without needing to understand each other's internal workings. For example, when you use a weather app on your phone, it sends a request through an API to a remote server, which processes the request and sends back the weather data in a structured format that the app can display. APIs are fundamental to modern software development because they enable developers to leverage existing services and functionalities—such as payment processing, mapping, or social media integration—without having to build everything from scratch, promoting efficiency, modularity, and interoperability across platforms and systems.**

- Input tokens: `15` | Output tokens: `114` | Cost: `$0.001622`


### claude-haiku-4-5
**An **API (Application Programming Interface)** is a set of rules, protocols, and tools that allows different software applications to communicate with each other. It acts as an intermediary, defining the methods and data formats that programs can use to request and exchange information, without needing to understand each other's internal workings. For example, when you use a weather app on your phone, it sends a request through an API to a remote server, which processes the request and sends back the weather data in a structured format that the app can display. APIs are fundamental to modern software development because they enable developers to leverage existing services and functionalities—such as payment processing, mapping, or social media integration—without having to build everything from scratch, promoting efficiency, modularity, and interoperability across platforms and systems.**

- Input tokens: `17` | Output tokens: `150` | Cost: `$0.000767`


### claude-opus-4-6
**An **API (Application Programming Interface)** is a set of rules, protocols, and tools that allows different software applications to communicate with each other. It acts as an intermediary, defining the methods and data formats that programs can use to request and exchange information, without needing to understand each other's internal workings. For example, when you use a weather app on your phone, it sends a request through an API to a remote server, which processes the request and sends back the weather data in a structured format that the app can display. APIs are fundamental to modern software development because they enable developers to leverage existing services and functionalities—such as payment processing, mapping, or social media integration—without having to build everything from scratch, promoting efficiency, modularity, and interoperability across platforms and systems.**

- Input tokens: `17` | Output tokens: `170` | Cost: `$0.004335`


---
### Multi-Turn Conversation with Cost Tracking

LiteLLM also supports multi-turn conversations. We can track the cumulative cost across turns.

In [15]:
model = "gpt-4o-mini"
total_cost = 0.0

conversation = [
    {"role": "user", "content": "My name is Alice."},
]

# Turn 1
response_turn1 = completion(model=model, messages=conversation, max_tokens=50)
assistant_reply1 = response_turn1.choices[0].message.content
total_cost += completion_cost(completion_response=response_turn1)

display(Markdown(f"""
**User:** My name is Alice.  
**Assistant:** {assistant_reply1}
"""))

# Turn 2
conversation.append({"role": "assistant", "content": assistant_reply1})
conversation.append({"role": "user", "content": "What is my name?"})

response_turn2 = completion(model=model, messages=conversation, max_tokens=50)
assistant_reply2 = response_turn2.choices[0].message.content
total_cost += completion_cost(completion_response=response_turn2)

display(Markdown(f"""
**User:** What is my name?  
**Assistant:** {assistant_reply2}

**Total conversation cost:** `${total_cost:.6f}`
"""))


**User:** My name is Alice.  
**Assistant:** Nice to meet you, Alice! How can I assist you today?



**User:** What is my name?  
**Assistant:** Your name is Alice.

**Total conversation cost:** `$0.000019`


---
### Cost Comparison for the Same Prompt

Let's send the same creative prompt to every model and compare costs side by side, sorted cheapest first.

In [16]:
# A haiku has 3 lines with a specific syllable pattern
# Line 1 → 5 syllables
# Line 2 → 7 syllables
# Line 3 → 5 syllables

prompt = "Write a haiku about programming."
results = []

# Run prompt across models
for model in MODELS:
    try:
        response = completion(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
        )
        cost = completion_cost(completion_response=response)
        reply = response.choices[0].message.content
        results.append({"model": model, "cost": cost, "reply": reply})
    except Exception as e:
        results.append({"model": model, "cost": None, "reply": f"Error: {e}"})

# Sort by cost (cheapest first)
results.sort(key=lambda x: x["cost"] if x["cost"] is not None else float("inf"))

# Display results using Markdown
display(Markdown("## Cost comparison for the same prompt"))

for r in results:
    cost_str = f"${r['cost']:.6f}" if r["cost"] is not None else "N/A"

    display(Markdown(f"""
### {r['model']}
- Cost: **{cost_str}**

**Response**
```text
{r['reply']}
```
"""))

## Cost comparison for the same prompt


### gpt-4o-mini
- Cost: **$0.000013**

**Response**
```text
Lines of code converge,  
Logic dances in my mind,  
Joy in every bug.
```



### claude-haiku-4-5
- Cost: **$0.000124**

**Response**
```text
Code flows like water,
Bugs hide in the darkness—
Debug brings the light.
```



### gpt-5.2
- Cost: **$0.000289**

**Response**
```text
Silent keys tap code  
Logic flows through midnight screens  
Bugs fade with sunrise
```



### claude-opus-4-6
- Cost: **$0.000770**

**Response**
```text
**Silent cursor blinks**
**Logic weaves through lines of code**
**Bugs hide in the dark**
```


---
## 2. Unified Call Function

To compare models fairly, we need a function that calls *any* provider and returns standardized results:

In [18]:
# Model registry for direct SDK calls (Sections 2-5)
MODEL_REGISTRY = {
    "gpt-4o-mini":       {"provider": "openai",    "input": 0.15,  "output": 0.60},
    "gpt-4o":            {"provider": "openai",    "input": 2.50,  "output": 10.00},
    "claude-sonnet-4-6": {"provider": "anthropic", "input": 3.00,  "output": 15.00},
    "claude-haiku-4-5":  {"provider": "anthropic", "input": 1.00,  "output": 5.00},
    "claude-opus-4-6":   {"provider": "anthropic", "input": 5.00,  "output": 25.00},
}

def estimate_cost(model, input_tokens, output_tokens):
    info = MODEL_REGISTRY[model]
    return (input_tokens / 1_000_000) * info["input"] + (output_tokens / 1_000_000) * info["output"]

def call_model(prompt: str, model: str, temperature: float = 0.7) -> dict:
    """Call any model and return standardized result dict."""
    info = MODEL_REGISTRY[model]
    start = time.perf_counter()

    if info["provider"] == "openai":
        r = openai_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200, temperature=temperature
        )
        text = r.choices[0].message.content
        in_tok, out_tok = r.usage.prompt_tokens, r.usage.completion_tokens

    elif info["provider"] == "anthropic":
        r = anthropic_client.messages.create(
            model=model, max_tokens=200, temperature=temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        text = r.content[0].text
        in_tok, out_tok = r.usage.input_tokens, r.usage.output_tokens

    elapsed = time.perf_counter() - start
    cost = estimate_cost(model, in_tok, out_tok)

    return {
        "model": model, "provider": info["provider"],
        "text": text, "input_tokens": in_tok, "output_tokens": out_tok,
        "latency_ms": round(elapsed * 1000), "cost_usd": cost
    }

---
## 3. Head-to-Head Comparison

Let's compare all available models on the **same prompt**:

In [ ]:
from IPython.display import display, Markdown

# Run comparison
test_prompt = "Explain what a REST API is to a junior developer. Be concise."

display(Markdown(f"##  Model Comparison\n**Prompt:** \"{test_prompt}\"\n---"))

results = []

for model_name in MODEL_REGISTRY:
    try:
        result = call_model(test_prompt, model_name)
        results.append(result)

        # IMPORTANT: do NOT wrap result['text'] in ``` or <pre>
        md = f"""
### 🤖 {model_name} ({result['provider']})

- **Latency:** `{result['latency_ms']} ms`
- **Tokens:** `{result['input_tokens']} in / {result['output_tokens']} out`
- **Cost:** `${result['cost_usd']:.6f}`

---

{result['text']}

---
"""
        display(Markdown(md))

    except Exception as e:
        display(Markdown(f"###  {model_name}\n`Error: {e}`"))

##  Model Comparison
**Prompt:** "Explain what a REST API is to a junior developer. Be concise."
---


### 🤖 gpt-4o-mini (openai)

- **Latency:** `3851 ms`
- **Tokens:** `21 in / 200 out`
- **Cost:** `$0.000123`

---

A REST API (Representational State Transfer Application Programming Interface) is a set of rules that allows different software applications to communicate over the web. It uses standard HTTP methods like GET, POST, PUT, and DELETE to perform operations on resources, which are typically represented in formats like JSON or XML.

Key points to understand:

1. **Resources**: Everything in a REST API is treated as a resource (e.g., users, posts). Each resource is identified by a unique URL.

2. **Stateless**: Each request from a client to the server must contain all the information needed to understand and process that request. The server does not store any client context between requests.

3. **HTTP Methods**:
   - **GET**: Retrieve data.
   - **POST**: Create new data.
   - **PUT**: Update existing data.
   - **DELETE**: Remove data.

4. **JSON/XML**: Data is typically exchanged in JSON format, which is lightweight and

---



### 🤖 gpt-4o (openai)

- **Latency:** `2062 ms`
- **Tokens:** `21 in / 91 out`
- **Cost:** `$0.000963`

---

A REST API (Representational State Transfer Application Programming Interface) is a way for different software applications to communicate over the internet. It uses HTTP requests to perform standard operations like GET (retrieve data), POST (send data), PUT (update data), and DELETE (remove data) on resources identified by URLs. REST APIs are stateless, meaning each request from a client contains all the information needed to process it, allowing for scalability and flexibility in web services.

---



### 🤖 claude-sonnet-4-6 (anthropic)

- **Latency:** `4995 ms`
- **Tokens:** `23 in / 199 out`
- **Cost:** `$0.003054`

---

## What is a REST API?

A **REST API** is a way for two applications to communicate over the internet using standard HTTP requests.

Think of it like a **waiter in a restaurant**:
- You (the client) make a **request**
- The waiter (the API) takes it to the kitchen (the server)
- The kitchen sends back a **response**

---

### The Basic Operations (HTTP Methods)

| Method | Action | Example |
|--------|--------|---------|
| `GET` | Read data | Fetch a user's profile |
| `POST` | Create data | Register a new user |
| `PUT` | Update data | Edit a profile |
| `DELETE` | Delete data | Remove a user |

---

### A Simple Example

You want to get a user from an API:

```
GET https://api.example.com/users/42

---



### 🤖 claude-haiku-4-5 (anthropic)

- **Latency:** `2707 ms`
- **Tokens:** `23 in / 200 out`
- **Cost:** `$0.001023`

---

# REST API Explained

A **REST API** is a set of rules for building web services that let different applications talk to each other over HTTP.

## Key Concepts

**REST** = Representational State Transfer — a style of architecture

**API** = Application Programming Interface — a way for programs to communicate

## How It Works

Instead of complicated custom protocols, REST uses standard HTTP methods:

- **GET** — retrieve data
- **POST** — create data
- **PUT** — update data
- **DELETE** — remove data

## Simple Example

```
GET /api/users/123
```
This fetches user #123 in a predictable, standard way.

## Why It's Popular

✅ Simple and intuitive  
✅ Uses HTTP (already everywhere)  
✅ Stateless (each request is independent)  
✅ Easy to test and debug  

**

---



### 🤖 claude-opus-4-6 (anthropic)

- **Latency:** `6056 ms`
- **Tokens:** `23 in / 200 out`
- **Cost:** `$0.005115`

---

# REST API — A Simple Explanation

A **REST API** is a way for two applications to communicate over the internet using standard **HTTP** methods — the same protocol your browser uses.

## Core Idea
Your app sends a **request** to a URL (called an **endpoint**), and the server sends back a **response** (usually JSON).

## HTTP Methods (CRUD)
| Method | Action | Example |
|--------|---------|---------|
| `GET` | Read data | Fetch a list of users |
| `POST` | Create data | Add a new user |
| `PUT/PATCH` | Update data | Edit a user's name |
| `DELETE` | Remove data | Delete a user |

## Quick Example
```
GET https://api.example.com/users/42
```
→ Returns the user with ID 42:
```json
{
  "

---


---
## 4. Summary Table

In [20]:
# ── Summary table ─────────────────────────────────────
if results:
    header =  "| Model | Provider | Latency | Tokens | Cost |\n"
    header += "|-------|----------|---------|--------|------|\n"
    rows = ""
    for r in results:
        rows += (f"| {r['model']} | {r['provider']} | {r['latency_ms']} ms | "
                 f"{r['input_tokens']}+{r['output_tokens']} | ${r['cost_usd']:.6f} |\n")
    display(Markdown(header + rows))

    # Cheapest and fastest
    cheapest = min(results, key=lambda x: x['cost_usd'])
    fastest  = min(results, key=lambda x: x['latency_ms'])
    print(f"\n Cheapest: {cheapest['model']} (${cheapest['cost_usd']:.6f})")
    print(f" Fastest:  {fastest['model']} ({fastest['latency_ms']} ms)")

| Model | Provider | Latency | Tokens | Cost |
|-------|----------|---------|--------|------|
| gpt-4o-mini | openai | 3851 ms | 21+200 | $0.000123 |
| gpt-4o | openai | 2062 ms | 21+91 | $0.000963 |
| claude-sonnet-4-6 | anthropic | 4995 ms | 23+199 | $0.003054 |
| claude-haiku-4-5 | anthropic | 2707 ms | 23+200 | $0.001023 |
| claude-opus-4-6 | anthropic | 6056 ms | 23+200 | $0.005115 |



 Cheapest: gpt-4o-mini ($0.000123)
 Fastest:  gpt-4o (2062 ms)


---
## 5. Multiple Test Prompts

A single prompt isn't enough — let's test across different task types:

In [ ]:
from IPython.display import display, Markdown

# ── Multi-test comparison ─────────────────────────────
test_suite = [
    ("Factual",    "What is the speed of light in km/s?"),
    ("Creative",   "Write a haiku about programming."),
    ("Analytical", "What are 3 pros and 3 cons of microservices?"),
    ("Reasoning",  "If all roses are flowers and some flowers fade quickly, can we conclude that some roses fade quickly?"),
]

for test_name, prompt in test_suite:
    display(Markdown(f"""
##  Test: {test_name}
**Prompt:** {prompt}
---
"""))

    for model_name in MODEL_REGISTRY:
        try:
            r = call_model(prompt, model_name, temperature=0.3)

            # Preview (first 200 chars) but keep it markdown-safe and readable
            preview = r["text"][:200] + ("…" if len(r["text"]) > 200 else "")
            preview = preview.replace("\n", " ")  # keep preview on one line

            display(Markdown(f"""
###  {r['model']}
- **Latency:** `{r['latency_ms']} ms`
- **Cost:** `${r['cost_usd']:.6f}`

**Preview:** {preview}

<details>
<summary>Show full response</summary>

{r['text']}

</details>

---
"""))
        except Exception as e:
            display(Markdown(f"###  {model_name}\n`Error: {e}`\n---"))

---
## 6. Beyond OpenAI & Anthropic: Alternative Providers

The OpenAI Python SDK is a **universal interface**. By changing just `base_url`,
you can use [Groq](https://groq.com) (speed-optimized) or [Ollama](https://ollama.com) (local, free).

This means: **learn one SDK, use everywhere**.

In [ ]:
# ── Alternative providers using the same SDK ─────────────
from openai import OpenAI

# Groq: cloud, ultra-fast inference
groq_client = None
if os.getenv("GROQ_API_KEY"):
    groq_client = OpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=os.getenv("GROQ_API_KEY")
    )
    MODEL_REGISTRY["llama-3.1-8b-instant"] = {"provider": "groq", "input": 0.05, "output": 0.08}
    print("Groq available (set GROQ_API_KEY in .env)")
else:
    print("Groq: Set GROQ_API_KEY in .env to enable")

# Ollama: local, completely free
ollama_client = None
try:
    import requests as _rq
    _rq.get("http://localhost:11434/api/tags", timeout=2)
    ollama_client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama"
    )
    MODEL_REGISTRY["llama3.2"] = {"provider": "ollama", "input": 0.0, "output": 0.0}
    print("Ollama available (local, free!)")
except:
    print("Ollama: Install from ollama.com and run 'ollama serve' to enable")

print(f"\nTotal models available: {len(MODEL_REGISTRY)}")
for name, info in MODEL_REGISTRY.items():
    print(f"  {name} ({info['provider']})")

In [ ]:
# ── Update call_model to support new providers ──────────

# Extend the original call_model function
_orig_call_model = call_model

def call_model(prompt: str, model: str, temperature: float = 0.7) -> dict:
    """Call any model including Groq and Ollama."""
    info = MODEL_REGISTRY[model]

    if info["provider"] in ("groq", "ollama"):
        c = groq_client if info["provider"] == "groq" else ollama_client
        if c is None:
            return {"model": model, "provider": info["provider"],
                    "text": f"[{info['provider']} not available]",
                    "input_tokens": 0, "output_tokens": 0,
                    "latency_ms": 0, "cost_usd": 0}
        start = time.perf_counter()
        r = c.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200, temperature=temperature
        )
        elapsed = time.perf_counter() - start
        in_tok = getattr(r.usage, 'prompt_tokens', 0) or 0
        out_tok = getattr(r.usage, 'completion_tokens', 0) or 0
        cost = estimate_cost(model, in_tok, out_tok)
        return {
            "model": model, "provider": info["provider"],
            "text": r.choices[0].message.content,
            "input_tokens": in_tok, "output_tokens": out_tok,
            "latency_ms": round(elapsed * 1000), "cost_usd": cost
        }

    return _orig_call_model(prompt, model, temperature)

# Quick test with all available providers
print("Quick test across all providers:\n")
for model_name in MODEL_REGISTRY:
    try:
        r = call_model("Say hello in 5 words.", model_name, temperature=0)
        print(f"  {r['model']:30} ({r['provider']:10}) {r['latency_ms']:5} ms | {r['text'][:50]}")
    except Exception as e:
        print(f"  {model_name:30} ERROR: {e}")

> **Key insight:** The OpenAI SDK is a *universal interface*. Groq, Ollama, Azure, and many
> other providers all support it. Learn one SDK, use it everywhere.

> **Exercise:** Add your own `GROQ_API_KEY` to the `.env` file (free at [console.groq.com](https://console.groq.com))
> and re-run the comparison. Notice the speed difference!

---
## Key Takeaways 📝

| Insight | Detail |
|---------|--------|
| **Speed ≠ Quality** | Faster models may give shorter, simpler answers |
| **Cost varies 10–100×** | gpt-4o-mini costs ~16× less than gpt-4o |
| **Use the cheapest model that works** | Start with mini/haiku, upgrade if quality is insufficient |
| **Multi-provider = resilience** | If one API is down, switch to another |
| **Always benchmark** | Don't assume — measure quality on *your* specific tasks |
| **Alternative providers** | Same SDK works with Groq, Ollama, and others |

---
**Next:** `05_token_explorer.ipynb` — Deep dive into tokenization and cost calculation